# Importing Libraries and Data

In [95]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [96]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [97]:
folder_path = '' #Add folder path here

In [98]:
#Importing Booking and Prices Data
spreadsheets = [file for file in os.listdir(f'{folder_path}/AirBnB Listings/Enriched Listings')]
cities = []

for spreadsheet in spreadsheets:
    filepath = f'{folder_path}/AirBnB Listings/Enriched Listings/{spreadsheet}'

    if '_listings_bookings_and_prices.csv' in spreadsheet:
        city_state = spreadsheet.replace('_listings_bookings_and_prices.csv', '')
        city = '_'.join(city_state.split('_')[:-1])  # drop the state abbreviation
        city = city.replace(' ', '_')  # handle spaces like "fort lauderdale"
        df_name = f'df_{city}_bookings_and_prices'
        cities.append(city)
    else:
        continue

    # Load CSV into a DataFrame with a dynamic variable name
    globals()[df_name] = pd.read_csv(filepath)
print(f'Import Complete')

Import Complete


In [99]:
#Importing POIs data
spreadsheets = [file for file in os.listdir(f'{folder_path}/POIs and City Centers')]

for spreadsheet in spreadsheets:
    filepath = f'{folder_path}/POIs and City Centers/{spreadsheet}'

    if '_pois.csv' in spreadsheet:
        city_state = spreadsheet.replace('_pois.csv', '')
        city = '_'.join(city_state.split('_')[:-1])  # drop the state abbreviation
        city = city.replace(' ', '_')  # handle spaces like "fort lauderdale"
        df_name = f'df_{city}_pois'

    else:
        continue

    # Load CSV into a DataFrame with a dynamic variable name
    print(spreadsheet)
    globals()[df_name] = pd.read_csv(filepath)

print(f'Import Complete')

austin_tx_pois.csv
cape coral_fl_pois.csv
charlotte_nc_pois.csv
dallas_tx_pois.csv
destin_fl_pois.csv
fort lauderdale_fl_pois.csv
houston_tx_pois.csv
jacksonville_fl_pois.csv
myrtle beach_sc_pois.csv
nashville_tn_pois.csv
phoenix_az_pois.csv
san antonio_tx_pois.csv
tampa_fl_pois.csv
atlanta_ga_pois.csv
indianapolis_in_pois.csv
miami_fl_pois.csv
Import Complete


# Data Preprocessing

Dropping observations with null value for any variable of interest (prices, bookings, coordinates, amenities, bedroom count)

In [100]:
for city in set(cities):
  print(f"{city}: {globals().get(f'df_{city}_bookings_and_prices').shape}")

  df_city = globals().get(f'df_{city}_bookings_and_prices')

  df_city = df_city[df_city.Bedrooms <= 7].copy() # Only including properties with bedroom count below 7
  df_city = df_city.dropna(subset=["Avg_Price", "Bookings", "Amenities", "Latitude", "Longitude", "Bedrooms"])

  globals()[f'df_{city}_bookings_and_prices'] = df_city

  print(f"{city} after dropping nulls: {globals().get(f'df_{city}_bookings_and_prices').shape}")

destin: (23052, 13)
destin after dropping nulls: (17164, 13)
phoenix: (14124, 13)
phoenix after dropping nulls: (6485, 13)
myrtle_beach: (19356, 13)
myrtle_beach after dropping nulls: (11214, 13)
houston: (52992, 13)
houston after dropping nulls: (26244, 13)
charlotte: (27084, 13)
charlotte after dropping nulls: (17649, 13)
indianapolis: (22176, 13)
indianapolis after dropping nulls: (11081, 13)
cape_coral: (24648, 13)
cape_coral after dropping nulls: (19239, 13)
austin: (45648, 13)
austin after dropping nulls: (25531, 13)
miami: (60672, 13)
miami after dropping nulls: (43070, 13)
nashville: (70572, 13)
nashville after dropping nulls: (43195, 13)
jacksonville: (22404, 13)
jacksonville after dropping nulls: (15737, 13)
atlanta: (47076, 13)
atlanta after dropping nulls: (24106, 13)
dallas: (20184, 13)
dallas after dropping nulls: (7852, 13)
san_antonio: (42972, 13)
san_antonio after dropping nulls: (21802, 13)
tampa: (35676, 13)
tampa after dropping nulls: (22577, 13)
fort_lauderdale: (5

Dropping data points where listings were priced very high to avoid bookings

In [101]:
## Sometimes price of the rentals are set exorbitantly high because owner doesn't want to rent it. These data points are discarded here.
# 1. Calculate global margins (same as your original logic)
all_prices_list = [globals().get(f'df_{city}_bookings_and_prices') for city in cities]
df_all_prices = pd.concat(all_prices_list)

df_global_margins = df_all_prices.groupby(['Bedrooms'], as_index=False)['Avg_Price'].agg(lower_margin=lambda x: x.quantile(0.05), upper_margin=lambda x: x.quantile(0.90))

## 2. Initialize the 4x4 Figure
#fig, axes = plt.subplots(4, 4, figsize=(24, 10), sharey=True)
#axes_flat = axes.flatten()  # Flatten to iterate easily

for i, city in enumerate(sorted(list(set(cities)))):
    if i >= 16: break # Limit to 16 plots for the 4x4 grid

    ax = axes_flat[i]

    # Data Processing
    print(f"{city} before cleaning: {globals().get(f'df_{city}_bookings_and_prices').shape}")
    df_city = globals().get(f'df_{city}_bookings_and_prices')
    df_city = pd.merge(df_city, df_global_margins, on='Bedrooms', how='left')

    # Apply filtering logic
    df_city = df_city[(df_city.Avg_Price < df_city.upper_margin) | (df_city.Bookings > 0)]

    globals()[f'df_{city}_bookings_and_prices'] = df_city
    print(f"{city} after cleaning: {globals().get(f'df_{city}_bookings_and_prices').shape}")

    #df_plot = df_city.sort_values('Bedrooms')
    #unique_margins = df_plot.drop_duplicates('Bedrooms').sort_values('Bedrooms')

    ## Plotting on the specific 'ax'
    #sns.boxplot(data=df_plot, x='Bedrooms', y='Avg_Price', color='lightgray', showfliers=False, width=0.6, ax=ax)

    ## Overlay Global Margins
    # We use np.arange and step logic to align with the boxplot's categorical x-axis (0, 1, 2...)
    #x_indices = np.arange(len(unique_margins))

    #ax.step(x_indices, unique_margins['upper_margin'], where='mid', color='red', linestyle='--', alpha=0.7, label='Global P90')
    #ax.step(x_indices, unique_margins['lower_margin'], where='mid', color='red', linestyle='--', alpha=0.7, label='Global P05')

    ## Shade the margin area
    #ax.fill_between(x_indices, unique_margins['lower_margin'], unique_margins['upper_margin'], step='mid', color='red', alpha=0.1)

    ## Formatting individual subplot
    #ax.set_title(f'{city.title()}', fontsize=14)
    #ax.set_ylim(0, 5000)
    #ax.set_xlabel('Bedrooms' if i >= 12 else '') # Only show x-label on bottom row
    #ax.set_ylabel('Price' if i % 4 == 0 else '') # Only show y-label on far left column
    #ax.grid(axis='y', linestyle=':', alpha=0.6)

## Hide unused subplots if there are fewer than 16 cities
#for j in range(i + 1, 16):
    #axes_flat[j].axis('off')

# Single Legend for the whole figure
#handles, labels = axes_flat[0].get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=12)

#plt.tight_layout()
#plt.show()

atlanta before cleaning: (24106, 13)
atlanta after cleaning: (23823, 15)
austin before cleaning: (25531, 13)
austin after cleaning: (24500, 15)
cape_coral before cleaning: (19239, 13)
cape_coral after cleaning: (18995, 15)
charlotte before cleaning: (17649, 13)
charlotte after cleaning: (17315, 15)
dallas before cleaning: (7852, 13)
dallas after cleaning: (7512, 15)
destin before cleaning: (17164, 13)
destin after cleaning: (16192, 15)
fort_lauderdale before cleaning: (37080, 13)
fort_lauderdale after cleaning: (35151, 15)
houston before cleaning: (26244, 13)
houston after cleaning: (25791, 15)
indianapolis before cleaning: (11081, 13)
indianapolis after cleaning: (10834, 15)
jacksonville before cleaning: (15737, 13)
jacksonville after cleaning: (15107, 15)
miami before cleaning: (43070, 13)
miami after cleaning: (41324, 15)
myrtle_beach before cleaning: (11214, 13)
myrtle_beach after cleaning: (10946, 15)
nashville before cleaning: (43195, 13)
nashville after cleaning: (41389, 15)
pho

Adding dummy variables for major amenity categories

In [102]:
df_categories = pd.read_csv(f'{folder_path}/AirBnB Listings/all_amenities_categorized.csv', encoding='cp1252')

# 2. Create a fast lookup dictionary: {'1 inch TV...': 'Electronics', 'Washer': 'Washer/Dryer', ...}
amenity_to_category = dict(zip(df_categories['Amenity'], df_categories['Category']))

# 3. Get a clean list of all unique categories
unique_categories = df_categories['Category'].unique().tolist()

# 4. Define a function to process a single row's string of amenities
def extract_categories(amenities_str):
    # Initialize all categories to 0
    categories_present = {cat: 0 for cat in unique_categories}

    if pd.isna(amenities_str):
        return categories_present

    # Convert the string "['Kitchen', 'Wifi']" back into a real Python list
    amenities_list = ast.literal_eval(amenities_str)

    # Loop through the list, find the category, and flip the 0 to a 1
    for amenity in amenities_list:
        category = amenity_to_category.get(amenity)
        if category: # If the amenity exists in our lookup mapping
            categories_present[category] = 1

    return categories_present

# 5. Loop through every city dataframe and apply the transformation
for city in set(cities):
  print(f"Adding dummy variables for amenities to {city}")
  df_city = globals().get(f'df_{city}_bookings_and_prices')

  # Drop duplicates to avoid redundant loops
  df_listings = df_city[['ID', 'Amenities']].drop_duplicates().reset_index(drop=True)

  # Apply the function to generate a column of dictionaries, then expand it into columns
  # This is much faster and cleaner than looping through rows manually
  dummies_df = df_listings['Amenities'].apply(extract_categories).apply(pd.Series)

  # Concatenate the new dummy columns to the original dataframe
  df_updated = pd.concat([df_listings, dummies_df], axis=1)
  print(f"{df_city.shape[0]} rows before merge")
  df_city = pd.merge(df_city, df_updated, how = 'left', on = ['ID', 'Amenities'])
  print(f"{df_city.shape[0]} rows after merge")

  # Save it back to the global variables
  globals()[f'df_{city}_bookings_and_prices'] = df_city

print("Category dummy variables have been successfully added to all city dataframes!")

Adding dummy variables for amenities to destin
16192 rows before merge
16192 rows after merge
Adding dummy variables for amenities to phoenix
6154 rows before merge
6154 rows after merge
Adding dummy variables for amenities to myrtle_beach
10946 rows before merge
10946 rows after merge
Adding dummy variables for amenities to houston
25791 rows before merge
25791 rows after merge
Adding dummy variables for amenities to charlotte
17315 rows before merge
17315 rows after merge
Adding dummy variables for amenities to indianapolis
10834 rows before merge
10834 rows after merge
Adding dummy variables for amenities to cape_coral
18995 rows before merge
18995 rows after merge
Adding dummy variables for amenities to austin
24500 rows before merge
24500 rows after merge
Adding dummy variables for amenities to miami
41324 rows before merge
41324 rows after merge
Adding dummy variables for amenities to nashville
41389 rows before merge
41389 rows after merge
Adding dummy variables for amenities to

Adding distance from city center

In [103]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    # Earth radius in meters
    R = 6371000

    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_city_center = pd.read_csv(f"{folder_path}/POIs and City Centers/city_centers.csv")

for city in cities:
    df_city = globals().get(f'df_{city}_bookings_and_prices')
    print(f"Adding distance from city center for {city} to {df_city.shape[0]} observations")

    # Get center coordinates
    city_center_lat = float(df_city_center.loc[df_city_center.city == city, "latitude"].iloc[0])
    city_center_lon = float(df_city_center.loc[df_city_center.city == city, "longitude"].iloc[0])

    df_city["city_center_lat"] = city_center_lat
    df_city["city_center_lon"] = city_center_lon

    # Calculate distance in meters using the Haversine function
    df_city["distance_from_city_center"] = haversine_vectorized(df_city["Latitude"], df_city["Longitude"], df_city["city_center_lat"], df_city["city_center_lon"])

    globals()[f'df_{city}_bookings_and_prices'] = df_city

print("All distances from city centers have been added successfully")

Adding distance from city center for jacksonville to 15107 observations
Adding distance from city center for houston to 25791 observations
Adding distance from city center for phoenix to 6154 observations
Adding distance from city center for myrtle_beach to 10946 observations
Adding distance from city center for nashville to 41389 observations
Adding distance from city center for tampa to 22281 observations
Adding distance from city center for fort_lauderdale to 35151 observations
Adding distance from city center for charlotte to 17315 observations
Adding distance from city center for indianapolis to 10834 observations
Adding distance from city center for destin to 16192 observations
Adding distance from city center for dallas to 7512 observations
Adding distance from city center for san_antonio to 21481 observations
Adding distance from city center for austin to 24500 observations
Adding distance from city center for cape_coral to 18995 observations
Adding distance from city center fo

In [104]:
for city in cities:
  df_city = globals().get(f'df_{city}_bookings_and_prices')
  df_city.to_csv(f"{folder_path}/AirBnB Listings/Cleaned Enriched Listings/df_{city}_bookings_and_prices.csv", index = False)